# Analysis

In [ ]:
import pandas as pd

area = pd.read_csv('Area.txt', sep='\t', index_col=0)
intensity = pd.read_csv('Intensity.txt', sep='\t', index_col=0)

In [ ]:
area.head()

In [ ]:
# Check data info
print("Shape of data:", area.shape)
print("\nUnique Tissues:", area['Tissue'].unique())
print("Unique Groups:", area['Group'].unique())
print("\nMissing values before filling:", area.isna().sum().sum())

area_filled = area.fillna(0)
intensity_filled = intensity.fillna(0)

In [ ]:
# Prepare data for PCA
# Separate metadata columns from numeric features
metadata_cols = ['Tissue', 'Mouse', 'Spot', 'Group']
numeric_cols = [col for col in area_filled.columns if col not in metadata_cols]

# Extract features (X) and labels
X = area_filled[numeric_cols].values
tissues = area_filled['Tissue'].values
groups = area_filled['Group'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Standardize the features (important for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Check explained variance
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

print(f"Variance explained by PC1: {explained_variance_ratio[0]:.2%}")
print(f"Variance explained by PC2: {explained_variance_ratio[1]:.2%}")
print(f"Cumulative variance (PC1 + PC2): {cumulative_variance[1]:.2%}")

In [ ]:
# Plot scree plot (variance explained by each PC)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 21), explained_variance_ratio[:20], 'bo-')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.title('Scree Plot (First 20 PCs)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, 21), cumulative_variance[:20], 'ro-')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance Explained')
plt.title('Cumulative Variance Explained')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create a DataFrame for easier plotting
pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'PC3': X_pca[:, 2],
    'Tissue': tissues,
    'Group': groups
})

pca_df.head()

In [ ]:
# PCA Plot colored by Tissue - to see if we can separate tissues
plt.figure(figsize=(12, 8))

# Define colors for tissues
tissue_colors = {
    'Blood serum': '#1f77b4',
    'Breast tissue': '#ff7f0e', 
    'Fur': '#2ca02c',
    'Liver tissue': '#d62728'
}

for tissue in pca_df['Tissue'].unique():
    mask = pca_df['Tissue'] == tissue
    plt.scatter(
        pca_df.loc[mask, 'PC1'], 
        pca_df.loc[mask, 'PC2'],
        label=tissue,
        c=tissue_colors[tissue],
        alpha=0.7,
        s=100,
        edgecolors='black',
        linewidth=0.5
    )

plt.xlabel(f'PC1 ({explained_variance_ratio[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({explained_variance_ratio[1]:.1%} variance)', fontsize=12)
plt.title('PCA: Separation of Tissues', fontsize=14, fontweight='bold')
plt.legend(title='Tissue', fontsize=10, title_fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA Plot with subplots for each tissue showing group distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

# Define markers for groups
group_markers = {'Cancer': 'o', 'Control': 's'}
group_colors = {'Cancer': '#e74c3c', 'Control': '#3498db'}

for idx, tissue in enumerate(sorted(pca_df['Tissue'].unique())):
    ax = axes[idx]
    tissue_data = pca_df[pca_df['Tissue'] == tissue]
    
    for group in tissue_data['Group'].unique():
        group_data = tissue_data[tissue_data['Group'] == group]
        ax.scatter(
            group_data['PC1'],
            group_data['PC2'],
            label=group,
            c=group_colors[group],
            marker=group_markers[group],
            alpha=0.7,
            s=120,
            edgecolors='black',
            linewidth=0.5
        )
    
    ax.set_xlabel(f'PC1 ({explained_variance_ratio[0]:.1%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({explained_variance_ratio[1]:.1%} variance)', fontsize=11)
    ax.set_title(f'{tissue}', fontsize=12, fontweight='bold')
    ax.legend(title='Group', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('PCA: Group Distribution within Each Tissue', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Combined plot showing both Tissue and Group information
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Colored by Tissue with shape indicating Group
ax1 = axes[0]
for tissue in sorted(pca_df['Tissue'].unique()):
    for group in sorted(pca_df['Group'].unique()):
        mask = (pca_df['Tissue'] == tissue) & (pca_df['Group'] == group)
        data = pca_df[mask]
        ax1.scatter(
            data['PC1'],
            data['PC2'],
            label=f'{tissue} - {group}',
            c=tissue_colors[tissue],
            marker=group_markers[group],
            alpha=0.7,
            s=100,
            edgecolors='black',
            linewidth=0.5
        )

ax1.set_xlabel(f'PC1 ({explained_variance_ratio[0]:.1%} variance)', fontsize=12)
ax1.set_ylabel(f'PC2 ({explained_variance_ratio[1]:.1%} variance)', fontsize=12)
ax1.set_title('PCA: Tissue (Color) & Group (Shape)', fontsize=13, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Colored by Group with text labels for Tissue
ax2 = axes[1]
for group in sorted(pca_df['Group'].unique()):
    mask = pca_df['Group'] == group
    ax2.scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        label=group,
        c=group_colors[group],
        alpha=0.6,
        s=100,
        edgecolors='black',
        linewidth=0.5
    )

ax2.set_xlabel(f'PC1 ({explained_variance_ratio[0]:.1%} variance)', fontsize=12)
ax2.set_ylabel(f'PC2 ({explained_variance_ratio[1]:.1%} variance)', fontsize=12)
ax2.set_title('PCA: Group Distribution', fontsize=13, fontweight='bold')
ax2.legend(title='Group', fontsize=10, title_fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA Plot to see all three dimensions
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(16, 6))

# 3D plot colored by Tissue
ax1 = fig.add_subplot(121, projection='3d')
for tissue in sorted(pca_df['Tissue'].unique()):
    mask = pca_df['Tissue'] == tissue
    ax1.scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        pca_df.loc[mask, 'PC3'],
        label=tissue,
        c=tissue_colors[tissue],
        alpha=0.7,
        s=60,
        edgecolors='black',
        linewidth=0.3
    )

ax1.set_xlabel(f'PC1 ({explained_variance_ratio[0]:.1%})', fontsize=10)
ax1.set_ylabel(f'PC2 ({explained_variance_ratio[1]:.1%})', fontsize=10)
ax1.set_zlabel(f'PC3 ({explained_variance_ratio[2]:.1%})', fontsize=10)
ax1.set_title('3D PCA: Tissue Separation', fontsize=12, fontweight='bold')
ax1.legend(title='Tissue', fontsize=8)

# 3D plot colored by Group
ax2 = fig.add_subplot(122, projection='3d')
for group in sorted(pca_df['Group'].unique()):
    mask = pca_df['Group'] == group
    ax2.scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        pca_df.loc[mask, 'PC3'],
        label=group,
        c=group_colors[group],
        marker=group_markers[group],
        alpha=0.7,
        s=60,
        edgecolors='black',
        linewidth=0.3
    )

ax2.set_xlabel(f'PC1 ({explained_variance_ratio[0]:.1%})', fontsize=10)
ax2.set_ylabel(f'PC2 ({explained_variance_ratio[1]:.1%})', fontsize=10)
ax2.set_zlabel(f'PC3 ({explained_variance_ratio[2]:.1%})', fontsize=10)
ax2.set_title('3D PCA: Group Distribution', fontsize=12, fontweight='bold')
ax2.legend(title='Group', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Print summary statistics
print("Sample counts by Tissue:")
print(pca_df['Tissue'].value_counts().sort_index())
print("\n" + "="*50 + "\n")

print("Sample counts by Group:")
print(pca_df['Group'].value_counts().sort_index())
print("\n" + "="*50 + "\n")

print("Sample counts by Tissue and Group:")
print(pd.crosstab(pca_df['Tissue'], pca_df['Group']))
print("\n" + "="*50 + "\n")

print(f"Total variance explained by first 3 PCs: {cumulative_variance[2]:.2%}")
print(f"Total variance explained by first 5 PCs: {cumulative_variance[4]:.2%}")
print(f"Total variance explained by first 10 PCs: {cumulative_variance[9]:.2%}")

## Summary Statistics

In [ ]:
# Replace NaN values with 0
area_filled = area.fillna(0)
print("Missing values after filling:", area_filled.isna().sum().sum())